# PCB Inspector - E01 Baseline (Leakage-Safe)
## Tujuan:
Melatih model YOLOv8n dengan split berbasis kelompok PCB (leakage-safe).
## Struktur Notebook:
1. Setup & Install
2. Deteksi Path
3. Konversi Dataset
4. Verifikasi
5. Training
6. Evaluasi

In [ ]:
# ==================================================
# 1. SETUP & INSTALL LIBRARY
# ==================================================
!pip install -q ultralytics pillow pyyaml

import os
import shutil
import random
import glob
from pathlib import Path
import xml.etree.ElementTree as ET
from PIL import Image

print("Library siap.")

In [ ]:
# ==================================================
# 2. DETEKSI PATH DATASET (Otomatis)
# ==================================================
# Mencari folder "images" dan "Annotations" di dalam /kaggle/input
images_dir = None
ann_dir = None

for root, dirs, files in os.walk("/kaggle/input"):
    if "images" in dirs and "Annotations" in dirs:
        images_dir = os.path.join(root, "images")
        ann_dir = os.path.join(root, "Annotations")
        break

if images_dir and ann_dir:
    INPUT_PATH = os.path.dirname(images_dir)
    print(f"✅ Path ditemukan!")
    print(f"Images      : {images_dir}")
    print(f"Annotations : {ann_dir}")
    print(f"Root        : {INPUT_PATH}")
else:
    print("❌ Path tidak ditemukan. Periksa struktur dataset.")

In [ ]:
# ==================================================
# 3. KONVERSI PASCAL VOC KE YOLO (Split Berbasis File)
# ==================================================
# --- Path Data ---
images_dir = Path(INPUT_PATH) / "images"
ann_dir = Path(INPUT_PATH) / "Annotations"

# --- Mapping Kelas ---
CLASS_MAP = {
    "missing_hole": 0, "mouse_bite": 1, "open_circuit": 2,
    "short": 3, "spur": 4, "spurious_copper": 5
}
CLASS_NAMES = list(CLASS_MAP.keys())

# --- Path Output ---
YOLO_DATASET = Path("/kaggle/working/yolo_dataset")
if YOLO_DATASET.exists():
    shutil.rmtree(YOLO_DATASET)
YOLO_DATASET.mkdir(parents=True, exist_ok=True)

# --- Helper: Membaca dan memindahkan file split jika belum ada di working ---
for split in ["train", "val", "test"]:
    src = Path(f"/kaggle/input/datasets/arkv99/kaggleworking/{split}.txt")
    dst = Path(f"/kaggle/working/{split}.txt")
    if not dst.exists() and src.exists():
        shutil.copy(src, dst)
        print(f"Memindahkan {split}.txt ke /kaggle/working...")

# --- Baca File Split ---
splits = {}
for split in ["train", "val", "test"]:
    split_file = Path(f"/kaggle/working/{split}.txt")
    if not split_file.exists():
        print(f"Warning: {split_file} tidak ditemukan, lewati")
        continue
    lines = split_file.read_text().strip().splitlines()
    splits[split] = [Path(line.replace("\\", "/")) for line in lines if line]
    print(f"Split {split}: {len(splits[split])} gambar")

# --- Fungsi Parse XML ---
def parse_xml(xml_path):
    tree = ET.parse(xml_path)
    root = tree.getroot()
    size = root.find("size")
    img_w = float(size.find("width").text) if size is not None else None
    img_h = float(size.find("height").text) if size is not None else None
    objects = []
    for obj in root.findall("object"):
        name = obj.find("name").text.strip()
        bbox = obj.find("bndbox")
        xmin = float(bbox.find("xmin").text)
        ymin = float(bbox.find("ymin").text)
        xmax = float(bbox.find("xmax").text)
        ymax = float(bbox.find("ymax").text)
        objects.append((name, xmin, ymin, xmax, ymax))
    return objects, img_w, img_h

# --- Proses Konversi ---
total_boxes = 0
for split_name, img_rel_paths in splits.items():
    print(f"Memproses split: {split_name}...")
    (YOLO_DATASET / split_name / "images").mkdir(parents=True, exist_ok=True)
    (YOLO_DATASET / split_name / "labels").mkdir(parents=True, exist_ok=True)
    
    for i, rel_path in enumerate(img_rel_paths):
        img_path = (INPUT_PATH / rel_path).resolve()
        if not img_path.exists():
            print(f"Missing image: {img_path}")
            continue
        stem = img_path.stem
        
        # Cari XML
        xml_path = None
        for candidate in ann_dir.rglob(f"{stem}.xml"):
            xml_path = candidate
            break
        if xml_path is None:
            print(f"Missing XML for {stem}")
            continue
        
        objects, img_w, img_h = parse_xml(xml_path)
        with open(YOLO_DATASET / split_name / "labels" / f"{stem}.txt", "w") as f:
            for name, xmin, ymin, xmax, ymax in objects:
                if name not in CLASS_MAP:
                    continue
                class_id = CLASS_MAP[name]
                if img_w is None or img_h is None:
                    with Image.open(img_path) as im:
                        img_w, img_h = im.size
                x_center = (xmin + xmax) / 2 / img_w
                y_center = (ymin + ymax) / 2 / img_h
                width = (xmax - xmin) / img_w
                height = (ymax - ymin) / img_h
                f.write(f"{class_id} {x_center:.6f} {y_center:.6f} {width:.6f} {height:.6f}\n")
                total_boxes += 1
        
        # Copy gambar
        shutil.copy2(img_path, YOLO_DATASET / split_name / "images" / img_path.name)
        
        # Progress Bar sederhana
        if i % 50 == 0:
            print(f"  {i}/{len(img_rel_paths)} gambar diproses...")

print(f"Selesai! Total boxes: {total_boxes}")

# --- Buat data.yaml ---
data_yaml = YOLO_DATASET / "data.yaml"
with open(data_yaml, "w") as f:
    f.write(f"path: {YOLO_DATASET.as_posix()}\n")
    f.write(f"train: {YOLO_DATASET.as_posix()}/train/images\n")
    f.write(f"val: {YOLO_DATASET.as_posix()}/val/images\n")
    f.write(f"test: {YOLO_DATASET.as_posix()}/test/images\n")
    f.write(f"nc: {len(CLASS_NAMES)}\n")
    f.write(f"names: {CLASS_NAMES}\n")

print("Data.yaml berhasil dibuat!")

In [ ]:
# ==================================================
# 4. VERIFIKASI DATASET YOLO
# ==================================================
train_dir = "/kaggle/working/yolo_dataset/train/images"
val_dir = "/kaggle/working/yolo_dataset/val/images"
test_dir = "/kaggle/working/yolo_dataset/test/images"

print(f"Jumlah gambar di Train: {len(os.listdir(train_dir))}")
print(f"Jumlah gambar di Val  : {len(os.listdir(val_dir))}")
print(f"Jumlah gambar di Test : {len(os.listdir(test_dir))}")
print(f"Total gambar         : {len(os.listdir(train_dir)) + len(os.listdir(val_dir)) + len(os.listdir(test_dir))}")

In [16]:
# ==================================================
# 5. TRAINING BASELINE E01 v2
# ==================================================
from ultralytics import YOLO

model = YOLO("yolov8n.pt")
results = model.train(
    data="/kaggle/working/yolo_dataset/data.yaml",
    epochs=50,
    imgsz=640,
    batch=16,
    device=0,
    project="experiments/E01_baseline_v2",   # Nama baru
    name="yolov8n_640",
    exist_ok=True,
    seed=42,
    pretrained=True,
    verbose=True
)

Ultralytics 8.4.138 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/yolo_dataset/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolov8n_640, nbs=64, nms

In [17]:
# ==================================================
# 6. EVALUASI MODEL & SIMPAN METRICS
# ==================================================
print("Evaluasi pada validation set...")
metrics = model.val(
    data="/kaggle/working/yolo_dataset/data.yaml",
    split="val",
    imgsz=640,
    batch=16,
    device=0
)

print("\n=== METRICS E01 v2 ===")
print(f"mAP50    : {metrics.box.map50:.4f}")
print(f"mAP50-95 : {metrics.box.map:.4f}")
print(f"Precision: {metrics.box.mp:.4f}")
print(f"Recall   : {metrics.box.mr:.4f}")

# F1 Rata-rata (Cara yang benar untuk Ultralytics v8.4+)
f1_mean = sum(metrics.box.f1) / len(metrics.box.f1)
print(f"F1       : {f1_mean:.4f}")

print("\n=== Per-Class AP ===")
for i, name in enumerate(['missing_hole', 'mouse_bite', 'open_circuit', 'short', 'spur', 'spurious_copper']):
    print(f"{name:<20}: AP50={metrics.box.ap50[i]:.4f}, AP50-95={metrics.box.ap[i]:.4f}")

# Simpan summary ke JSON
import json
summary = {
    "experiment": "E01_baseline_v2",
    "model": "yolov8n",
    "imgsz": 640,
    "epochs": 50,
    "batch": 16,
    "seed": 42,
    "mAP50": metrics.box.map50,
    "mAP50_95": metrics.box.map,
    "precision": metrics.box.mp,
    "recall": metrics.box.mr,
    "f1": f1_mean
}
with open("/kaggle/working/E01_summary.json", "w") as f:
    json.dump(summary, f, indent=2)
print("\nSummary tersimpan di /kaggle/working/E01_summary.json")

Evaluasi pada validation set...
Ultralytics 8.4.138 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
Model summary (fused): 73 layers, 3,006,818 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 4642.2±806.9 MB/s, size: 1390.8 KB)
val: Scanning /kaggle/working/yolo_dataset/val/labels.cache... 120 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 120/120 50.3Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 8/8 1.9it/s 4.3s0.3ss
                   all        120        358      0.377       0.39      0.362      0.135
          missing_hole         20         61      0.316      0.967      0.962      0.452
            mouse_bite         20         53      0.369      0.245      0.146     0.0477
          open_circuit         20         59      0.502      0.288       0.26     0.0617
                 short         20         72      0.601      0.514      0.557      0.169
 

In [22]:
# ==================================================
# E02 — AUGMENTATION BASELINE (640px)
# ==================================================
from ultralytics import YOLO

model = YOLO("yolov8n.pt")
results = model.train(
    data="/kaggle/working/yolo_dataset/data.yaml",
    epochs=50,
    imgsz=640,
    batch=16,
    device=0,
    project="experiments/E02_augmentation",
    name="yolov8n_aug",
    exist_ok=True,
    seed=42,
    pretrained=True,
    verbose=True,
    # Augmentation Settings (diubah dari default)
    hsv_h=0.015,       # Hue
    hsv_s=0.7,         # Saturation
    hsv_v=0.4,         # Value
    degrees=10.0,      # Rotation (maksimal 10 derajat)
    translate=0.1,     # Translation
    scale=0.5,         # Scaling
    shear=2.0,         # Shear
    perspective=0.0,
    flipud=0.5,        # Flip Up-Down (0.5 artinya 50% chance)
    fliplr=0.5,        # Flip Left-Right
    mosaic=1.0,        # Mosaic (1.0 artinya 100% aktif)
    mixup=0.0          # Mixup (biarkan 0 untuk deteksi objek murni)
)

Ultralytics 8.4.138 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/yolo_dataset/data.yaml, degrees=10.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.5, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolov8n_aug, nbs=64, nm

In [18]:
# ==================================================
# E03 — HIGHER RESOLUTION TRAINING (imgsz=1280)
# ==================================================
from ultralytics import YOLO

model = YOLO("yolov8n.pt")
results = model.train(
    data="/kaggle/working/yolo_dataset/data.yaml",
    epochs=50,
    imgsz=1280,  # Meningkatkan resolusi dari 640 ke 1280
    batch=8,      # Turunkan batch karena resolusi lebih tinggi
    device=0,
    project="experiments/E03_higher_res",
    name="yolov8n_1280",
    exist_ok=True,
    seed=42,
    pretrained=True,
    verbose=True
)

Ultralytics 8.4.138 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/yolo_dataset/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1280, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolov8n_1280, nbs=64, nm

In [20]:
# ==================================================
# E04 — TILING STRATEGY (PERSIAPAN DATASET)
# ==================================================
import cv2
import numpy as np
import os, shutil
from pathlib import Path
from PIL import Image

# --- Konstanta ---
TILE_SIZE = 640  # Ukuran tile (harus sama dengan yang dipakai di fungsi)

# --- Path Dataset YOLO Lama ---
YOLO_DATASET = Path("/kaggle/working/yolo_dataset")
TILED_DATASET = Path("/kaggle/working/tiled_dataset")

# Hapus folder lama jika ada
if TILED_DATASET.exists():
    shutil.rmtree(TILED_DATASET)
TILED_DATASET.mkdir(parents=True, exist_ok=True)

# --- Fungsi Tiling ---
def tile_image(image_path, tile_size=TILE_SIZE):
    """Bagi gambar menjadi tile-tile berukuran tile_size x tile_size."""
    img = cv2.imread(str(image_path))
    h, w = img.shape[:2]
    
    # Hitung jumlah tile
    n_h = h // tile_size
    n_w = w // tile_size
    
    # Jika gambar terlalu kecil, skip
    if n_h == 0 or n_w == 0:
        return None, None, None
    
    tiles = []
    coords = []
    
    for i in range(n_h):
        for j in range(n_w):
            y1, y2 = i * tile_size, (i + 1) * tile_size
            x1, x2 = j * tile_size, (j + 1) * tile_size
            tile = img[y1:y2, x1:x2]
            tiles.append(tile)
            coords.append((x1, y1, x2, y2))
    
    return tiles, coords, (w, h)

# --- Proses Setiap Split ---
for split in ["train", "val", "test"]:
    src_images = YOLO_DATASET / split / "images"
    src_labels = YOLO_DATASET / split / "labels"
    dst_images = TILED_DATASET / split / "images"
    dst_labels = TILED_DATASET / split / "labels"
    
    dst_images.mkdir(parents=True, exist_ok=True)
    dst_labels.mkdir(parents=True, exist_ok=True)
    
    print(f"Memproses split: {split}...")
    
    for img_file in src_images.iterdir():
        if img_file.suffix.lower() not in ['.jpg', '.jpeg', '.png', '.bmp']:
            continue
        
        # Baca gambar
        img_path = str(img_file)
        tiles, coords, (w, h) = tile_image(img_path)
        
        if tiles is None:
            continue
        
        # Baca label YOLO
        label_file = src_labels / f"{img_file.stem}.txt"
        labels = []
        if label_file.exists():
            with open(label_file, "r") as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) == 5:
                        class_id = int(parts[0])
                        x_center = float(parts[1]) * w
                        y_center = float(parts[2]) * h
                        box_w = float(parts[3]) * w
                        box_h = float(parts[4]) * h
                        labels.append((class_id, x_center, y_center, box_w, box_h))
        
        # Untuk setiap tile, simpan gambar dan label yang sesuai
        for idx, (tile, (x1, y1, x2, y2)) in enumerate(zip(tiles, coords)):
            # Simpan tile sebagai gambar
            tile_name = f"{img_file.stem}_tile{idx}.jpg"
            cv2.imwrite(str(dst_images / tile_name), tile)
            
            # Buat label untuk tile (hanya objek yang berada di dalam tile ini)
            tile_labels = []
            for cls, xc, yc, bw, bh in labels:
                # Konversi koordinat absolut ke tile
                x_abs = xc
                y_abs = yc
                x_tile = x_abs - x1
                y_tile = y_abs - y1
                
                # Cek apakah objek berada di dalam tile
                # Menggunakan TILE_SIZE (global) bukan tile_size lokal
                if 0 <= x_tile < TILE_SIZE and 0 <= y_tile < TILE_SIZE:
                    # Konversi kembali ke koordinat YOLO relatif
                    x_center_norm = x_tile / TILE_SIZE
                    y_center_norm = y_tile / TILE_SIZE
                    w_norm = bw / TILE_SIZE
                    h_norm = bh / TILE_SIZE
                    
                    # Batasi agar tidak melebihi 1
                    x_center_norm = min(1, max(0, x_center_norm))
                    y_center_norm = min(1, max(0, y_center_norm))
                    w_norm = min(1, max(0, w_norm))
                    h_norm = min(1, max(0, h_norm))
                    
                    tile_labels.append(f"{cls} {x_center_norm:.6f} {y_center_norm:.6f} {w_norm:.6f} {h_norm:.6f}")
            
            # Simpan label tile
            with open(dst_labels / f"{tile_name.replace('.jpg', '.txt')}", "w") as f:
                f.write("\n".join(tile_labels))

print("Tiling selesai!")

# --- Buat data.yaml untuk tiled dataset ---
data_yaml_tiled = TILED_DATASET / "data.yaml"
with open(data_yaml_tiled, "w") as f:
    f.write(f"path: {TILED_DATASET.as_posix()}\n")
    f.write(f"train: {TILED_DATASET.as_posix()}/train/images\n")
    f.write(f"val: {TILED_DATASET.as_posix()}/val/images\n")
    f.write(f"test: {TILED_DATASET.as_posix()}/test/images\n")
    f.write(f"nc: 6\n")
    f.write(f"names: ['missing_hole', 'mouse_bite', 'open_circuit', 'short', 'spur', 'spurious_copper']\n")

print("data.yaml untuk tiled dataset berhasil dibuat!")

Memproses split: train...
Memproses split: val...
Memproses split: test...
Tiling selesai!
data.yaml untuk tiled dataset berhasil dibuat!


In [21]:
# ==================================================
# E04 — TRAINING YOLOv8n DENGAN TILED DATASET
# ==================================================
from ultralytics import YOLO

model = YOLO("yolov8n.pt")
results = model.train(
    data="/kaggle/working/tiled_dataset/data.yaml",
    epochs=50,
    imgsz=640,          # Tile berukuran 640
    batch=16,
    device=0,
    project="experiments/E04_tiling",
    name="yolov8n_tiled",
    exist_ok=True,
    seed=42,
    pretrained=True,
    verbose=True
)

Ultralytics 8.4.138 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/tiled_dataset/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolov8n_tiled, nbs=64, 

In [27]:
# ==================================================
# E04 — EVALUASI FINAL (INFERENCE MERGING + METRIK)
# ==================================================
import cv2
import numpy as np
import torch
import torchvision.ops as tv_ops
import json
import yaml
from pathlib import Path
from ultralytics import YOLO
from tqdm import tqdm
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

# --- Konfigurasi ---
MODEL_PATH = "/kaggle/working/runs/detect/experiments/E04_tiling/yolov8n_tiled/weights/best.pt"
DATA_YAML = "/kaggle/working/yolo_dataset/data.yaml"
TILE_SIZE = 640
CONF_THRESHOLD = 0.25
IOU_THRESHOLD = 0.45

# --- Load Model ---
model = YOLO(MODEL_PATH)

# --- Load Dataset Info ---
with open(DATA_YAML, 'r') as f:
    data_info = yaml.safe_load(f)

val_images_dir = Path(data_info['val'])
val_labels_dir = Path("/kaggle/working/yolo_dataset/val/labels")

# --- Fungsi Tiling Inference ---
def tile_inference(image_path):
    img = cv2.imread(str(image_path))
    h, w = img.shape[:2]
    all_boxes, all_scores, all_classes = [], [], []
    
    n_h = h // TILE_SIZE
    n_w = w // TILE_SIZE
    
    if n_h == 0 or n_w == 0:
        results = model.predict(img, conf=CONF_THRESHOLD, verbose=False)[0]
        for box in results.boxes:
            all_boxes.append(box.xyxy.cpu().numpy()[0])
            all_scores.append(box.conf.cpu().numpy()[0])
            all_classes.append(box.cls.cpu().numpy()[0])
    else:
        for i in range(n_h):
            for j in range(n_w):
                y1, y2 = i * TILE_SIZE, (i + 1) * TILE_SIZE
                x1, x2 = j * TILE_SIZE, (j + 1) * TILE_SIZE
                tile = img[y1:y2, x1:x2]
                results = model.predict(tile, conf=CONF_THRESHOLD, verbose=False)[0]
                for box in results.boxes:
                    bx1, by1, bx2, by2 = box.xyxy.cpu().numpy()[0]
                    bx1 += x1; by1 += y1; bx2 += x1; by2 += y1
                    all_boxes.append([bx1, by1, bx2, by2])
                    all_scores.append(box.conf.cpu().numpy()[0])
                    all_classes.append(box.cls.cpu().numpy()[0])
    
    if len(all_boxes) > 0:
        boxes_tensor = torch.tensor(all_boxes)
        scores_tensor = torch.tensor(all_scores)
        keep = tv_ops.nms(boxes_tensor, scores_tensor, iou_threshold=IOU_THRESHOLD)
        if len(keep) > 0:
            return (boxes_tensor[keep].cpu().numpy(),
                    scores_tensor[keep].cpu().numpy(),
                    torch.tensor(all_classes)[keep].cpu().numpy())
    return [], [], []

# --- Buat COCO Format ---
image_files = sorted(val_images_dir.glob("*"))
image_files = [f for f in image_files if f.suffix.lower() in ['.jpg', '.jpeg', '.png']]

images_info = []
annotations = []
predictions = []
image_id_map = {}

for idx, img_file in enumerate(image_files):
    image_id_map[img_file.stem] = idx
    img = cv2.imread(str(img_file))
    images_info.append({"id": idx, "file_name": img_file.name,
                        "width": img.shape[1], "height": img.shape[0]})

# Ground truth
gt_ann_id = 0
for img_file in image_files:
    label_file = val_labels_dir / f"{img_file.stem}.txt"
    if label_file.exists():
        with open(label_file, "r") as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) == 5:
                    class_id = int(parts[0]) + 1  # COCO 1-6
                    x_center, y_center, w, h = map(float, parts[1:])
                    img_w = images_info[image_id_map[img_file.stem]]["width"]
                    img_h = images_info[image_id_map[img_file.stem]]["height"]
                    x1 = (x_center - w/2) * img_w
                    y1 = (y_center - h/2) * img_h
                    x2 = (x_center + w/2) * img_w
                    y2 = (y_center + h/2) * img_h
                    annotations.append({"id": gt_ann_id, "image_id": image_id_map[img_file.stem],
                                        "category_id": class_id,
                                        "bbox": [float(x1), float(y1), float(x2-x1), float(y2-y1)],
                                        "area": float((x2-x1)*(y2-y1)), "iscrowd": 0})
                    gt_ann_id += 1

# Predictions (TAMBAHKAN AREA!)
print("Melakukan inference...")
pred_id = 0
for img_file in tqdm(image_files, desc="Inference"):
    boxes, scores, classes = tile_inference(img_file)
    image_id = image_id_map[img_file.stem]
    for box, score, cls in zip(boxes, scores, classes):
        x1, y1, x2, y2 = box
        predictions.append({
            "id": pred_id,
            "image_id": image_id,
            "category_id": int(cls) + 1,
            "bbox": [float(x1), float(y1), float(x2 - x1), float(y2 - y1)],
            "area": float((x2 - x1) * (y2 - y1)),  # <--- TAMBAHKAN INI
            "score": float(score)
        })
        pred_id += 1

# --- Evaluasi COCO ---
coco_gt = COCO()
coco_gt.dataset = {"images": images_info, "annotations": annotations,
                   "categories": [{"id": i+1, "name": name} for i, name in enumerate(
                       ['missing_hole', 'mouse_bite', 'open_circuit', 'short', 'spur', 'spurious_copper'])]}
coco_gt.createIndex()

coco_dt = COCO()
coco_dt.dataset = {"images": images_info, "annotations": predictions,
                   "categories": coco_gt.dataset["categories"]}
coco_dt.createIndex()

coco_eval = COCOeval(coco_gt, coco_dt, iouType="bbox")
coco_eval.evaluate()
coco_eval.accumulate()
coco_eval.summarize()

# --- Print Hasil ---
print("\n" + "="*50)
print("HASIL E04 FINAL (PADA GAMBAR ASLI)")
print("="*50)
# Perbaikan: stats[0] adalah mAP@0.5:0.95, stats[1] adalah mAP@0.5
print(f"mAP50    : {coco_eval.stats[1]:.4f}")
print(f"mAP50-95 : {coco_eval.stats[0]:.4f}")
print(f"Precision: {coco_eval.stats[1]:.4f}")
print(f"Recall   : {coco_eval.stats[2]:.4f}")

print("\n=== Per-Class AP50 ===")
for i in range(1, 7):
    class_name = coco_gt.dataset["categories"][i-1]["name"]
    ap50 = coco_eval.eval["precision"][0, :, i-1, 0, 2]
    ap50 = ap50[~np.isnan(ap50)].mean() if len(ap50) > 0 else 0
    print(f"{class_name:<20}: {ap50:.4f}")

# Simpan JSON
results_summary = {
    "experiment": "E04_final_original_res",
    "model": "yolov8n_tiled",
    "mAP50": coco_eval.stats[1],
    "mAP50_95": coco_eval.stats[0],
    "precision": coco_eval.stats[1],
    "recall": coco_eval.stats[2],
    "per_class": {coco_gt.dataset["categories"][i-1]["name"]:
                  float(coco_eval.eval["precision"][0, :, i-1, 0, 2][~np.isnan(coco_eval.eval["precision"][0, :, i-1, 0, 2])].mean())
                  for i in range(1, 7)}
}
with open("/kaggle/working/E04_final_summary.json", "w") as f:
    json.dump(results_summary, f, indent=2)
print("\nSummary tersimpan di /kaggle/working/E04_final_summary.json")

Melakukan inference...


Inference: 100%|██████████| 120/120 [00:12<00:00,  9.76it/s]

creating index...
index created!
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=0.05s).
Accumulating evaluation results...
DONE (t=0.03s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.205
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.458
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.150
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.174
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.205
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.215
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.123
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.252
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.252
 Average Recall     (AR) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.195
 Av

In [29]:
!cp -r "/kaggle/working/runs/detect/experiments/E03_higher_res/yolov8n_1280" "/kaggle/working/E03_final_model"

In [30]:
# Pindahkan folder ke /kaggle/working (sudah ada, tapi ini memastikan)
import shutil
shutil.make_archive("/kaggle/working/E03_final_model", 'zip', "/kaggle/working/E03_final_model")

'/kaggle/working/E03_final_model.zip'